# Observador: U-Net para filamentos solares

Entrena una red que predice, píxel a píxel, la probabilidad de filamento. Después la convierte en
filamentos individuales, la mide con la **PQ oficial** en una validación propia y genera
`submission.csv`. **Este notebook no envía nada a la competición.**

Requisitos en Kaggle: *Accelerator* = GPU T4 o P100, *Internet* = activado (para instalar la librería
y bajar el encoder preentrenado), y la competición añadida como *Input*.

In [ ]:
# Configuración. PRUEBA_RAPIDA=True reduce todo para comprobar que el notebook funciona.
PRUEBA_RAPIDA = False

RES = 1024            # resolución de trabajo (la original es 2048)
ENCODER = "resnet34"
PESOS = "imagenet"   # encoder preentrenado (necesita Internet en Kaggle)
EPOCAS = 40
LOTE = 4
LR = 3e-4
VALIDAR_CADA = 5
SEMILLA = 42

if PRUEBA_RAPIDA:
    RES, EPOCAS, LOTE, VALIDAR_CADA, PESOS = 256, 2, 2, 1, None

In [ ]:
import subprocess, sys
try:
    import segmentation_models_pytorch as smp
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch"], check=True)
    import segmentation_models_pytorch as smp
try:
    import pycocotools
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pycocotools"], check=True)

## Código del proyecto
Copiado del repositorio `filamentos-solares-2026`.

In [ ]:
import os
os.makedirs('filamentos', exist_ok=True)

In [ ]:
%%writefile filamentos/__init__.py
"""Código del proyecto de segmentación de filamentos solares."""

In [ ]:
%%writefile filamentos/datos.py
"""Lectura de MAGFiLO y conversión entre polígonos, máscaras y RLE."""
import json
import os
from collections import defaultdict

import numpy as np
from pycocotools import mask as mask_utils

TAM = 2048


def cargar_anotaciones(raiz):
    """Devuelve {nombre_imagen: {anotador: [anotación COCO, ...]}}.

    Una imagen puede tener hasta 3 registros, uno por anotador. El id del registro
    es "GGGGAA-<nombre>"; usamos GGGGAA como identificador del anotador.
    """
    with open(os.path.join(raiz, "train", "MAGFiLO_1.0_Annotations_kaggle2026_train.json")) as f:
        coco = json.load(f)
    registro = {i["id"]: (i["file_name"], i["id"].split("-")[0]) for i in coco["images"]}
    out = defaultdict(lambda: defaultdict(list))
    for reg_id, (nombre, anot) in registro.items():
        out[nombre][anot]  # imágenes anotadas sin filamentos también cuentan
    for a in coco["annotations"]:
        nombre, anot = registro[a["image_id"]]
        out[nombre][anot].append(a)
    return {k: dict(v) for k, v in out.items()}


def poligono_a_mascara(segmentacion, alto=TAM, ancho=TAM):
    rles = mask_utils.frPyObjects(segmentacion, alto, ancho)
    return mask_utils.decode(mask_utils.merge(rles)).astype(bool)


def mascaras_de(anotaciones):
    return [poligono_a_mascara(a["segmentation"]) for a in anotaciones]


def mascara_a_rle(mascara):
    """Cadena de 'counts' RLE de COCO, el formato de la columna segmentation_rle."""
    rle = mask_utils.encode(np.asfortranarray(mascara.astype(np.uint8)))
    return rle["counts"].decode("ascii")


def rle_a_mascara(counts, alto=TAM, ancho=TAM):
    return mask_utils.decode({"size": [alto, ancho], "counts": counts.encode("ascii")}).astype(bool)

In [ ]:
%%writefile filamentos/metrica.py
"""Panoptic Quality (PQ) tal como la define la competición, más diagnósticos.

Un filamento predicho y uno real se emparejan si su IoU > 0.5. Con ese umbral
cada filamento solo puede emparejarse con uno del otro lado, así que no hace
falta un emparejamiento óptimo.

    PQ = (suma de IoU de los aciertos / TP) * (TP / (TP + FP/2 + FN/2))
"""
from dataclasses import dataclass, field

import numpy as np

UMBRAL_IOU = 0.5


@dataclass
class Resultado:
    tp: int = 0
    fp: int = 0
    fn: int = 0
    suma_iou: float = 0.0
    # Diagnósticos que la rúbrica también mira
    uno_a_muchos: int = 0   # un filamento real cubierto por 2+ predichos (fragmentación)
    muchos_a_uno: int = 0   # un predicho que cubre 2+ reales (fusión)
    ious: list = field(default_factory=list)

    @property
    def sq(self):
        return self.suma_iou / self.tp if self.tp else 0.0

    @property
    def rq(self):
        den = self.tp + 0.5 * self.fp + 0.5 * self.fn
        return self.tp / den if den else 1.0

    @property
    def pq(self):
        return self.sq * self.rq

    def __iadd__(self, otro):
        for k in ("tp", "fp", "fn", "suma_iou", "uno_a_muchos", "muchos_a_uno"):
            setattr(self, k, getattr(self, k) + getattr(otro, k))
        self.ious += otro.ious
        return self


def _matriz_solape(reales, predichas):
    """Intersecciones y áreas usando etiquetas, sin comparar máscara contra máscara."""
    if not reales or not predichas:
        return np.zeros((len(reales), len(predichas))), None, None
    forma = reales[0].shape
    area_r = np.array([m.sum() for m in reales], float)
    area_p = np.array([m.sum() for m in predichas], float)
    inter = np.zeros((len(reales), len(predichas)))
    for i, r in enumerate(reales):
        # Recorte a la caja de r para no recorrer la imagen entera
        ys, xs = np.nonzero(r)
        y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
        rr = r[y0:y1, x0:x1]
        for j, p in enumerate(predichas):
            inter[i, j] = np.logical_and(rr, p[y0:y1, x0:x1]).sum()
    return inter, area_r, area_p


def evaluar_imagen(reales, predichas):
    """reales, predichas: listas de máscaras booleanas 2D del mismo tamaño."""
    res = Resultado()
    inter, area_r, area_p = _matriz_solape(reales, predichas)
    if area_r is None:
        res.fn, res.fp = len(reales), len(predichas)
        return res
    iou = inter / (area_r[:, None] + area_p[None, :] - inter)
    emparejados_r = set()
    emparejados_p = set()
    for i, j in zip(*np.nonzero(iou > UMBRAL_IOU)):
        emparejados_r.add(i)
        emparejados_p.add(j)
        res.suma_iou += iou[i, j]
        res.ious.append(float(iou[i, j]))
    res.tp = len(emparejados_r)
    res.fn = len(reales) - res.tp
    res.fp = len(predichas) - len(emparejados_p)

    # Un solape cuenta si cubre al menos el 10 % del filamento más pequeño de la pareja
    toca = inter > 0.1 * np.minimum(area_r[:, None], area_p[None, :])
    res.uno_a_muchos = int((toca.sum(axis=1) >= 2).sum())
    res.muchos_a_uno = int((toca.sum(axis=0) >= 2).sum())
    return res


def resumen(res):
    return (f"PQ {res.pq:.4f} (forma SQ {res.sq:.3f} x detección RQ {res.rq:.3f}) | "
            f"TP {res.tp} FP {res.fp} FN {res.fn} | "
            f"fragmentados {res.uno_a_muchos} fusionados {res.muchos_a_uno}")

In [ ]:
%%writefile filamentos/clasico.py
"""Línea base clásica: observador (zonas oscuras) + juez (reglas observadas).

Sin aprendizaje automático. Sirve para estrenar el formato de envío y como
referencia contra la que medir los modelos.
"""
from dataclasses import dataclass

import numpy as np
from PIL import Image
from scipy import ndimage as ndi
from skimage import measure, morphology


@dataclass
class Parametros:
    umbral_oscuro: float = 0.88    # intensidad relativa al fondo local
    radio_max: float = 0.90        # regla 2: nada más allá del 90 % del radio
    area_min: int = 330            # regla 3: tamaño mínimo
    oscuridad_min: float = 0.44    # regla 4: lo más negro son manchas solares
    alargamiento_min: float = 1.8  # regla 5: forma alargada
    union_px: int = 6              # trozos a menos de esta distancia se unen en un filamento


def disco_solar(img):
    """Centro y radio del disco a partir de los píxeles brillantes."""
    m = img > 0.3 * np.percentile(img, 99)
    m = ndi.binary_fill_holes(ndi.binary_opening(m, iterations=3))
    lab, _ = ndi.label(m)
    k = np.argmax(np.bincount(lab.ravel())[1:]) + 1
    yy, xx = np.nonzero(lab == k)
    return xx.mean(), yy.mean(), np.sqrt(len(xx) / np.pi)


def normalizar(img, cx, cy, R):
    """Divide por el fondo local (mediana en ventanas de ~100 px)."""
    yy, xx = np.indices(img.shape)
    r = np.hypot(xx - cx, yy - cy) / R
    dentro = r < 1
    relleno = np.where(dentro, img, np.median(img[dentro]))
    peque = np.asarray(Image.fromarray(relleno.astype(np.float32)).resize((512, 512), Image.BILINEAR))
    fondo = ndi.median_filter(peque, size=25)
    fondo = np.asarray(Image.fromarray(fondo).resize(img.shape[::-1], Image.BILINEAR))
    norm = img / np.maximum(fondo, 1)
    norm[~dentro] = 1
    return norm, r


def observador(norm, r, p):
    """Candidatos: zonas oscuras dentro del disco, uniendo trozos muy cercanos."""
    oscuro = (norm < p.umbral_oscuro) & (r < p.radio_max)
    oscuro = ndi.binary_opening(oscuro, iterations=1)
    # Los trozos cercanos se agrupan con una dilatación, pero la máscara final es la original
    grupos = measure.label(ndi.binary_dilation(oscuro, structure=morphology.disk(p.union_px)))
    return grupos * oscuro


def juez(etiquetas, norm, p):
    """Aplica las reglas a cada candidato y devuelve las máscaras aceptadas."""
    aceptadas = []
    for reg in measure.regionprops(etiquetas):
        if reg.area < p.area_min:
            continue
        y0, x0, y1, x1 = reg.bbox
        vals = norm[y0:y1, x0:x1][reg.image]
        if vals.min() < p.oscuridad_min:
            continue
        if reg.axis_major_length / max(reg.axis_minor_length, 1) < p.alargamiento_min:
            continue
        m = np.zeros(etiquetas.shape, bool)
        m[y0:y1, x0:x1] = reg.image
        aceptadas.append(m)
    return aceptadas


def predecir(ruta_imagen, p=Parametros()):
    img = np.asarray(Image.open(ruta_imagen).convert("L"), float)
    cx, cy, R = disco_solar(img)
    norm, r = normalizar(img, cx, cy, R)
    return juez(observador(norm, r, p), norm, p)

In [ ]:
%%writefile filamentos/observador.py
"""Observador con red neuronal: U-Net que predice, píxel a píxel, la probabilidad de filamento.

Entrada de 3 canales a resolución reducida:
  0. imagen original normalizada a [0, 1]
  1. imagen dividida por su fondo local (resalta lo oscuro respecto al entorno)
  2. distancia al centro del disco en radios (ayuda con la regla del borde)
Objetivo: media de las máscaras de todos los anotadores (0, 1/3, 1/2, 2/3 o 1).
"""
import numpy as np
import torch
from PIL import Image, ImageDraw
from scipy import ndimage as ndi

from . import clasico


def preparar_imagen(ruta, res):
    """Devuelve (entrada [3,res,res] float32, r [res,res])."""
    img = np.asarray(Image.open(ruta).convert("L").resize((res, res), Image.BILINEAR), np.float32)
    cx, cy, R = clasico.disco_solar(img)
    yy, xx = np.indices(img.shape)
    r = np.hypot(xx - cx, yy - cy) / R
    dentro = r < 1
    relleno = np.where(dentro, img, np.median(img[dentro]))
    fondo = ndi.median_filter(relleno, size=max(3, res // 20))
    norm = np.clip(img / np.maximum(fondo, 1), 0, 1.5)
    norm[~dentro] = 1
    entrada = np.stack([img / 255.0, norm / 1.5, np.clip(r, 0, 1.2) / 1.2]).astype(np.float32)
    return entrada, r.astype(np.float32)


def objetivo_suave(anotadores, res, tam=2048):
    """Media de las máscaras de cada anotador, dibujadas directamente a resolución res."""
    escala = res / tam
    acum = np.zeros((res, res), np.float32)
    for anns in anotadores.values():
        lienzo = Image.new("L", (res, res), 0)
        d = ImageDraw.Draw(lienzo)
        for a in anns:
            for p in a["segmentation"]:
                if len(p) >= 6:
                    d.polygon([(p[i] * escala, p[i + 1] * escala) for i in range(0, len(p), 2)], fill=1)
        acum += np.asarray(lienzo, np.float32)
    return acum / max(len(anotadores), 1)


class DatosEntreno(torch.utils.data.Dataset):
    def __init__(self, entradas, objetivos, aumentar):
        self.x, self.y, self.aumentar = entradas, objetivos, aumentar

    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        x, y = self.x[i], self.y[i][None]
        if self.aumentar:
            k = np.random.randint(4)
            x, y = np.rot90(x, k, (1, 2)), np.rot90(y, k, (1, 2))
            if np.random.rand() < 0.5:
                x, y = x[:, :, ::-1], y[:, :, ::-1]
            x = x.copy()
            x[0] = np.clip(x[0] * np.random.uniform(0.85, 1.15) + np.random.uniform(-0.05, 0.05), 0, 1)
        return torch.from_numpy(np.ascontiguousarray(x)), torch.from_numpy(np.ascontiguousarray(y))


def perdida(logits, y):
    """BCE + Dice suave: BCE aprende píxel a píxel, Dice compensa que el filamento ocupa poco."""
    bce = torch.nn.functional.binary_cross_entropy_with_logits(logits, y)
    p = torch.sigmoid(logits)
    inter = (p * y).sum((2, 3))
    dice = 1 - (2 * inter + 1) / (p.sum((2, 3)) + y.sum((2, 3)) + 1)
    return bce + dice.mean()


@torch.no_grad()
def predecir_prob(modelo, entrada, dispositivo, tta=True):
    """Probabilidad [res,res]; con TTA promedia las 8 rotaciones/volteos."""
    x = torch.from_numpy(entrada)[None].to(dispositivo)
    vistas = [(k, f) for k in range(4) for f in (False, True)] if tta else [(0, False)]
    acum = 0
    for k, f in vistas:
        xi = torch.rot90(x, k, (2, 3))
        if f:
            xi = xi.flip(3)
        p = torch.sigmoid(modelo(xi))
        if f:
            p = p.flip(3)
        acum = acum + torch.rot90(p, -k, (2, 3))
    return (acum / len(vistas))[0, 0].float().cpu().numpy()


def a_instancias(prob, r, umbral=0.5, area_min=330, area_max=60000, radio_max=0.95, union_px=0, tam=2048):
    """Probabilidad a resolución reducida -> lista de máscaras booleanas a 2048x2048."""
    prob_g = np.asarray(Image.fromarray(prob).resize((tam, tam), Image.BILINEAR))
    r_g = np.asarray(Image.fromarray(r).resize((tam, tam), Image.BILINEAR))
    # Solo dentro del disco: fuera no hay filamentos y una red poco entrenada puede dar ruido
    fil = (prob_g > umbral) & (r_g < 1.0)
    if union_px:
        grupos, n = ndi.label(ndi.binary_dilation(fil, iterations=union_px))
        etiquetas = grupos * fil
    else:
        etiquetas, n = ndi.label(fil, structure=np.ones((3, 3)))
    mascaras = []
    for k, sl in enumerate(ndi.find_objects(etiquetas), start=1):
        if sl is None:
            continue
        m_local = etiquetas[sl] == k
        area = m_local.sum()
        # El filamento anotado más grande mide ~38 000 px; algo mucho mayor es un error
        if area < area_min or area > area_max:
            continue
        if r_g[sl][m_local].mean() > radio_max:
            continue
        m = np.zeros((tam, tam), bool)
        m[sl] = m_local
        mascaras.append(m)
    return mascaras

## Datos y separación entrenamiento / validación

In [ ]:
import glob, hashlib, json, time, random
import numpy as np, torch
from multiprocessing import Pool
from filamentos.datos import cargar_anotaciones, mascaras_de, mascara_a_rle, rle_a_mascara
from filamentos.metrica import Resultado, evaluar_imagen, resumen
from filamentos import observador as O

random.seed(SEMILLA); np.random.seed(SEMILLA); torch.manual_seed(SEMILLA)

json_path = glob.glob('/kaggle/input/**/MAGFiLO_1.0_Annotations_kaggle2026_train.json', recursive=True)
if not json_path:  # ejecución fuera de Kaggle
    json_path = glob.glob(os.environ.get('MAGFILO', '.') + '/**/MAGFiLO_1.0_Annotations_kaggle2026_train.json', recursive=True)
DATOS = os.path.dirname(os.path.dirname(json_path[0]))
print('Datos en', DATOS)

ANOT = cargar_anotaciones(DATOS)
nombres = sorted(n for n in ANOT if os.path.exists(f'{DATOS}/train/train_images/{n}'))

# Validación fija: ~15 % de las imágenes, elegidas por hash del nombre (reproducible)
def es_val(n):
    return int(hashlib.md5(n.encode()).hexdigest(), 16) % 100 < 15
val = [n for n in nombres if es_val(n)]
tr = [n for n in nombres if not es_val(n)]
if PRUEBA_RAPIDA:
    tr, val = tr[:16], val[:6]
print(len(tr), 'entrenamiento,', len(val), 'validación')

In [ ]:
def preparar(n):
    x, r = O.preparar_imagen(f'{DATOS}/train/train_images/{n}', RES)
    return x, r, O.objetivo_suave(ANOT[n], RES)

t = time.time()
with Pool(os.cpu_count()) as p:
    cache = dict(zip(tr + val, p.map(preparar, tr + val)))
print(f'Preprocesado en {time.time() - t:.0f} s')

## Entrenamiento

In [ ]:
disp = 'cuda' if torch.cuda.is_available() else 'cpu'
modelo = smp.Unet(ENCODER, encoder_weights=PESOS, in_channels=3, classes=1).to(disp)
opt = torch.optim.AdamW(modelo.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCAS * max(1, len(tr) // LOTE))
escalador = torch.amp.GradScaler(enabled=disp == 'cuda')

dl = torch.utils.data.DataLoader(
    O.DatosEntreno([cache[n][0] for n in tr], [cache[n][2] for n in tr], aumentar=True),
    batch_size=LOTE, shuffle=True, drop_last=True, num_workers=2)

def evaluar(modelo, nombres, tta=False, **post):
    modelo.eval()
    tot = Resultado()
    for n in nombres:
        x, r, _ = cache[n]
        with torch.autocast(disp, enabled=disp == 'cuda'):
            prob = O.predecir_prob(modelo, x, disp, tta=tta)
        pred = O.a_instancias(prob, r, **post)
        for anns in ANOT[n].values():
            tot += evaluar_imagen(mascaras_de(anns), pred)
    return tot

mejor, historia = -1, []
for epoca in range(1, EPOCAS + 1):
    modelo.train(); t = time.time(); suma = 0
    for x, y in dl:
        x, y = x.to(disp), y.to(disp)
        with torch.autocast(disp, enabled=disp == 'cuda'):
            l = O.perdida(modelo(x), y)
        opt.zero_grad(); escalador.scale(l).backward(); escalador.step(opt); escalador.update(); sched.step()
        suma += l.item()
    linea = f'época {epoca}: pérdida {suma / len(dl):.4f} ({time.time() - t:.0f} s)'
    if epoca % VALIDAR_CADA == 0 or epoca == EPOCAS:
        res = evaluar(modelo, val)
        historia.append((epoca, res.pq))
        linea += ' | ' + resumen(res)
        if res.pq > mejor:
            mejor = res.pq
            torch.save(modelo.state_dict(), 'observador_mejor.pt')
    print(linea)
print('Mejor PQ de validación:', round(mejor, 4))

## Ajuste del juez en validación
Umbral de probabilidad, tamaño mínimo y unión de trozos cercanos.

In [ ]:
modelo.load_state_dict(torch.load('observador_mejor.pt', map_location=disp))
probs = {}
for n in val:
    x, r, _ = cache[n]
    with torch.autocast(disp, enabled=disp == 'cuda'):
        probs[n] = O.predecir_prob(modelo, x, disp, tta=True)

def pq_con(**post):
    tot = Resultado()
    for n in val:
        pred = O.a_instancias(probs[n], cache[n][1], **post)
        for anns in ANOT[n].values():
            tot += evaluar_imagen(mascaras_de(anns), pred)
    return tot

resultados = []
for umbral in (0.3, 0.4, 0.5, 0.6):
    for area_min in (150, 330, 600):
        for union_px in (0, 8):
            res = pq_con(umbral=umbral, area_min=area_min, union_px=union_px)
            resultados.append((res.pq, dict(umbral=umbral, area_min=area_min, union_px=union_px)))
            print(f'{umbral} {area_min} {union_px} -> {resumen(res)}')
MEJOR_POST = max(resultados, key=lambda t: t[0])[1]
print('Mejor post-proceso:', MEJOR_POST, 'PQ', round(max(resultados, key=lambda t: t[0])[0], 4))

## Predicción del test y `submission.csv`
El archivo se genera pero **no se envía**.

In [ ]:
import csv
test_dir = f'{DATOS}/test/test_images'
filas = []
for f in sorted(os.listdir(test_dir)):
    x, r = O.preparar_imagen(f'{test_dir}/{f}', RES)
    with torch.autocast(disp, enabled=disp == 'cuda'):
        prob = O.predecir_prob(modelo, x, disp, tta=True)
    img_id = os.path.splitext(f)[0]
    for k, m in enumerate(O.a_instancias(prob, r, **MEJOR_POST), start=1):
        filas.append((f'{img_id}_{k}', mascara_a_rle(m)))
    if PRUEBA_RAPIDA and len(filas) > 20:
        break
with open('submission.csv', 'w', newline='') as fh:
    w = csv.writer(fh)
    w.writerow(['filament_id', 'segmentation_rle'])
    w.writerows(filas)

# Comprobación del formato: ids únicos y RLE que se decodifica a 2048x2048
ids = [a for a, _ in filas]
assert len(ids) == len(set(ids))
if filas:
    assert rle_a_mascara(filas[0][1]).shape == (2048, 2048)
else:
    print('AVISO: el modelo no encontró ningún filamento en el test')
print(len(filas), 'filamentos en', len(set(i.rsplit('_', 1)[0] for i in ids)), 'imágenes -> submission.csv')